## Data Preparation


In [41]:
import pandas as pd


X_train = pd.read_csv("../datasets/X_train.csv")
y_train = pd.read_csv("../datasets/y_train.csv")

X_test = pd.read_csv("../datasets/X_test.csv")
y_test = pd.read_csv("../datasets/y_test.csv")

### Merging the columns of X_train + y_train and X_test + y_test

In [22]:
train_df = X_train.merge(y_train,on="id")

test_df = X_test.merge(y_test,on="id")

print(train_df.shape)
print(test_df.shape)

display(train_df.head())
display(test_df.head())

(1572, 3)
(657, 3)


,id,text,category_truth
0,1919,File Share Access - [TICKET ID] - [NAME]. [NAM...,Fileservice
1,1584,File Share Access - [TICKET ID] - [NAME] ([COM...,Fileservice
2,584,[TICKET ID] - A Support Ticket was forwarded t...,Support general
3,1393,[TICKET ID] - A Support Ticket was forwarded t...,Support general
4,8636,LVS not respondingLVS will not except my passw...,Fileservice


,id,text,category_truth
0,830,[TICKET ID] - New Support Ticket received - S...,Support general
1,1030,[TICKET ID] - New Support Ticket received - R...,Fileservice
2,1184,[TICKET ID] - New Support Ticket received - U...,Software
3,5570,The following system must be set to End Of Lif...,EOL
4,4585,File Share Access - [TICKET ID] - [NAME] ([COM...,Fileservice


### Ensuring that no NULL values exist and count of categories

In [23]:
print("Training data missing values:")
print(train_df[["text", "category_truth"]].isnull().sum())

print("\nTesting data missing values:")
print(test_df[["text", "category_truth"]].isnull().sum())

print("\nTraining category distribution:")
print(train_df["category_truth"].value_counts())

print("\nTesting category distribution:")
print(test_df["category_truth"].value_counts())

Training data missing values:
text              0
category_truth    0
dtype: int64

Testing data missing values:
text              0
category_truth    0
dtype: int64

Training category distribution:
category_truth
Fileservice          546
Support general      541
Software             164
O365                 127
Active Directory      84
Computer-Services     65
EOL                   45
Name: count, dtype: int64

Testing category distribution:
category_truth
Support general      221
Fileservice          138
O365                  92
Software              58
EOL                   58
Active Directory      49
Computer-Services     41
Name: count, dtype: int64


### Splitting No.of tickets for Training and Testing

In [24]:
X_train_text = train_df["text"].astype(str)
y_train = train_df["category_truth"]

X_test_text = test_df["text"].astype(str)
y_test = test_df["category_truth"]

print("Training tickets:", len(X_train_text))
print("Testing tickets:", len(X_test_text))
print("Categories:", y_train.unique())

Training tickets: 1572
Testing tickets: 657
Categories: ['Fileservice' 'Support general' 'Software' 'EOL' 'O365'
 'Active Directory' 'Computer-Services']


## Model Building


In [25]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

baseline_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_features=20000
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

baseline_model.fit(X_train_text, y_train)

# print("Baseline model trained successfully!")

,steps,"[('tfidf', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


## Model accuracy 


In [26]:
from sklearn.metrics import accuracy_score,classification_report,f1_score

y_pred = baseline_model.predict(X_test_text)

print("Test accuracy:", round(accuracy_score(y_test, y_pred), 4))

print("Macro F1-score:", round(f1_score(y_test, y_pred, average="macro"), 4))

print("\nClassification report:\n")

print(classification_report(y_test,y_pred,zero_division=0))

Test accuracy: 0.7428
Macro F1-score: 0.657

Classification report:

                   precision    recall  f1-score   support

 Active Directory       1.00      0.20      0.34        49
Computer-Services       0.89      0.41      0.57        41
              EOL       1.00      1.00      1.00        58
      Fileservice       1.00      0.91      0.95       138
             O365       0.79      0.52      0.63        92
         Software       0.57      0.28      0.37        58
  Support general       0.60      0.96      0.74       221

         accuracy                           0.74       657
        macro avg       0.84      0.61      0.66       657
     weighted avg       0.79      0.74      0.72       657



### Initial Check of model prediction

In [27]:
results = test_df[["text", "category_truth"]].copy()

results["predicted_category"] = y_pred
results["correct"] = (results["category_truth"] ==results["predicted_category"])

display(results.head(10))

,text,category_truth,predicted_category,correct
0,[TICKET ID] - New Support Ticket received - S...,Support general,Support general,True
1,[TICKET ID] - New Support Ticket received - R...,Fileservice,Support general,False
2,[TICKET ID] - New Support Ticket received - U...,Software,Support general,False
3,The following system must be set to End Of Lif...,EOL,EOL,True
4,File Share Access - [TICKET ID] - [NAME] ([COM...,Fileservice,Fileservice,True
5,The following system must be set to End Of Lif...,EOL,EOL,True
6,[TICKET ID] - A Support Ticket was forwarded t...,Software,Support general,False
7,[TICKET ID] - A Support Ticket was forwarded t...,Support general,Support general,True
8,[TICKET ID] - New Support Ticket received - L...,Support general,Support general,True
9,[TICKET ID] - New Support Ticket received - F...,Fileservice,Fileservice,True


In [28]:
errors = results[results["correct"] == False].copy()

print("Incorrect predictions:", len(errors))
print("\nMost common incorrect predictions:")

display(
    errors.groupby(
        ["category_truth", "predicted_category"]).size().sort_values(ascending=False).head(15).rename("count").reset_index())

Incorrect predictions: 169

Most common incorrect predictions:


,category_truth,predicted_category,count
0,O365,Support general,44
1,Active Directory,Support general,36
2,Software,Support general,36
3,Computer-Services,Support general,18
4,Fileservice,Support general,8
5,Software,O365,5
6,Support general,Software,5
7,Computer-Services,Software,4
8,Active Directory,O365,2
9,Fileservice,Software,2


In [29]:
from sklearn.metrics import confusion_matrix
import pandas as pd

labels = sorted(y_test.unique())

cm = confusion_matrix(y_test, y_pred, labels=labels)

cm_df = pd.DataFrame(
    cm,
    index=labels,
    columns=labels
)

display(cm_df)

print("Total test tickets:", cm.sum())
print("Correct predictions:", cm.trace())
print("Incorrect predictions:", cm.sum() - cm.trace())

,Active Directory,Computer-Services,EOL,Fileservice,O365,Software,Support general
Active Directory,10,0,0,0,2,1,36
Computer-Services,0,17,0,0,2,4,18
EOL,0,0,58,0,0,0,0
Fileservice,0,0,0,126,2,2,8
O365,0,0,0,0,48,0,44
Software,0,1,0,0,5,16,36
Support general,0,1,0,0,2,5,213


Total test tickets: 657
Correct predictions: 488
Incorrect predictions: 169


In [30]:
import numpy as np

tfidf = baseline_model.named_steps["tfidf"]
classifier = baseline_model.named_steps["classifier"]

feature_names = tfidf.get_feature_names_out()
categories = classifier.classes_

for category in categories:
    class_index = list(categories).index(category)
    coefficients = classifier.coef_[class_index]

    top_indices = np.argsort(coefficients)[-10:][::-1]

    print(f"\nCategory: {category}")
    print([
        (feature_names[i], round(coefficients[i], 3))
        for i in top_indices
    ])


Category: Active Directory
[('user', np.float64(2.258)), ('ad', np.float64(2.149)), ('löschen', np.float64(1.455)), ('needed', np.float64(1.25)), ('change', np.float64(1.238)), ('name', np.float64(1.155)), ('in location', np.float64(1.154)), ('usuário', np.float64(1.085)), ('password', np.float64(0.995)), ('reset', np.float64(0.966))]

Category: Computer-Services
[('impressora', np.float64(3.867)), ('printer', np.float64(3.507)), ('impresora', np.float64(1.501)), ('instalação', np.float64(1.258)), ('scanner', np.float64(1.205)), ('computador', np.float64(1.093)), ('de impressora', np.float64(1.058)), ('laptop', np.float64(0.793)), ('instalación', np.float64(0.762)), ('printers', np.float64(0.76))]

Category: EOL
[('reservations', np.float64(1.296)), ('if', np.float64(1.24)), ('and', np.float64(1.226)), ('remove', np.float64(1.122)), ('the', np.float64(1.092)), ('nexthink', np.float64(1.025)), ('please', np.float64(0.927)), ('please remove', np.float64(0.877)), ('any', np.float64(0.866

In [31]:
ticket = errors.iloc[0]["text"]
actual_category = errors.iloc[0]["category_truth"]

predicted_category = baseline_model.predict([ticket])[0]

ticket_vector = tfidf.transform([ticket])
class_index = list(categories).index(predicted_category)

contributions = (
    ticket_vector.toarray()[0]
    * classifier.coef_[class_index]
)

top_indices = np.argsort(contributions)[-10:][::-1]

print("Ticket:", ticket)
print("Actual category:", actual_category)
print("Predicted category:", predicted_category)
print("\nTop positive word contributions:")

for i in top_indices:
    if contributions[i] > 0:
        print(feature_names[i], round(contributions[i], 4))

Ticket: [TICKET ID] - New Support Ticket received  - R Drive Permission
Actual category: Fileservice
Predicted category: Support general

Top positive word contributions:
ticket 0.0963
id new 0.088
received 0.0681
permission 0.0639
ticket received 0.0512
new support 0.0512
support ticket 0.0442
ticket id 0.0332
new 0.0307
support 0.0304


In [32]:
def clean_ticket_template(text):
    text = str(text).lower()
    remove_phrases = [
        "new support ticket received",
        "a support ticket was forwarded to your role",
        "[ticket id]", "[name]", "[company]", "[location]", "[mac_address]",
        "[username]", "[id]"  # Added these two remaining placeholders
    ]
    for phrase in remove_phrases:
        text = text.replace(phrase, " ")
    return " ".join(text.split())

cleaned_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_features=20000,
        sublinear_tf=True,          # Boosts accuracy by smoothing repeated words
        preprocessor=clean_ticket_template
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        C=10.0,                     # Allows stronger keyword weights
        class_weight="balanced",    # Fixes minority categories like Active Directory
        random_state=42
    ))
])

cleaned_model.fit(train_df["text"], train_df["category_truth"])
cleaned_pred = cleaned_model.predict(test_df["text"])

In [33]:
from sklearn.metrics import accuracy_score, f1_score

comparison = pd.DataFrame([
    {
        "Model": "Original baseline",
        "Accuracy": accuracy_score(y_test, y_pred),
        "Macro F1": f1_score(
            y_test, y_pred, average="macro"
        )
    },
    {
        "Model": "Template-cleaned",
        "Accuracy": accuracy_score(y_test, cleaned_pred),
        "Macro F1": f1_score(
            y_test, cleaned_pred, average="macro"
        )
    }
])

display(comparison.round(4))

print("\nCleaned model classification report:")
print(classification_report(
    y_test,
    cleaned_pred,
    zero_division=0
))

,Model,Accuracy,Macro F1
0,Original baseline,0.7428,0.657
1,Template-cleaned,0.8052,0.763



Cleaned model classification report:
                   precision    recall  f1-score   support

 Active Directory       0.49      0.59      0.54        49
Computer-Services       0.70      0.68      0.69        41
              EOL       1.00      1.00      1.00        58
      Fileservice       0.97      0.93      0.95       138
             O365       0.78      0.70      0.74        92
         Software       0.62      0.57      0.59        58
  Support general       0.81      0.85      0.83       221

         accuracy                           0.81       657
        macro avg       0.77      0.76      0.76       657
     weighted avg       0.81      0.81      0.81       657



In [34]:
ticket = (
    "[TICKET ID] - New Support Ticket received "
    "- R Drive Permission"
)

print("Original model:", baseline_model.predict([ticket])[0])
print("Cleaned model:", cleaned_model.predict([ticket])[0])
print("Expected category: Fileservice")

Original model: Support general
Cleaned model: Fileservice
Expected category: Fileservice


In [35]:
cleaned_results = test_df[["text", "category_truth"]].copy()

cleaned_results["predicted_category"] = cleaned_pred
cleaned_results["correct"] = (
    cleaned_results["category_truth"]
    == cleaned_results["predicted_category"]
)

cleaned_errors = cleaned_results[
    ~cleaned_results["correct"]
]

print("Remaining errors:", len(cleaned_errors))

display(
    cleaned_errors.groupby(
        ["category_truth", "predicted_category"]
    ).size().sort_values(ascending=False)
    .head(15)
    .rename("count")
    .reset_index()
)

Remaining errors: 128


,category_truth,predicted_category,count
0,O365,Support general,18
1,Support general,Active Directory,15
2,Active Directory,Support general,14
3,Software,O365,9
4,Support general,Software,9
5,Software,Support general,6
6,O365,Active Directory,5
7,Software,Active Directory,5
8,Software,Computer-Services,5
9,Computer-Services,Software,4


In [36]:
def make_incomplete(text):
    words = str(text).split()
    half = len(words) // 2
    return " ".join(words[:half])


def introduce_typo(text):
    words = str(text).split()
    
    for i in range(len(words)):
        # Find the first word with 5+ letters and swap its 2nd and 3rd letters
        if len(words[i]) >= 5 and words[i].isalpha():
            w = words[i]
            words[i] = w[0] + w[2] + w[1] + w[3:]
            break
            
    return " ".join(words)


original_texts = test_df["text"].tolist()
incomplete_texts = [make_incomplete(t) for t in original_texts]
noisy_texts = [introduce_typo(t) for t in original_texts]

print("Original:", original_texts[0])
print("\nIncomplete:", incomplete_texts[0])
print("\nNoisy:", noisy_texts[0])

Original: [TICKET ID] - New Support Ticket received  - Setup BB on mobile phone.

Incomplete: [TICKET ID] - New Support Ticket

Noisy: [TICKET ID] - New Spuport Ticket received - Setup BB on mobile phone.


In [37]:
from sklearn.metrics import accuracy_score, f1_score

conditions = {
    "Original": original_texts,
    "Incomplete": incomplete_texts,
    "Noisy": noisy_texts
}

models = {
    "Original baseline": baseline_model,
    "Template-cleaned": cleaned_model
}

robustness_results = []

for model_name, model in models.items():
    for condition, texts in conditions.items():
        predictions = model.predict(texts)

        robustness_results.append({
            "Model": model_name,
            "Condition": condition,
            "Accuracy": accuracy_score(y_test, predictions),
            "Macro F1": f1_score(
                y_test,
                predictions,
                average="macro",
                zero_division=0
            )
        })

robustness_df = pd.DataFrame(robustness_results)

display(
    robustness_df.round(4).sort_values(
        ["Model", "Condition"]
    )
)

,Model,Condition,Accuracy,Macro F1
1,Original baseline,Incomplete,0.5967,0.4687
2,Original baseline,Noisy,0.7397,0.6608
0,Original baseline,Original,0.7428,0.6570
4,Template-cleaned,Incomplete,0.5175,0.4477
5,Template-cleaned,Noisy,0.7869,0.7457
3,Template-cleaned,Original,0.8052,0.7630


In [38]:
import numpy as np
import pandas as pd

# Probability assigned to each category
probabilities = cleaned_model.predict_proba(test_df["text"])

# Highest probability for each ticket
max_confidence = probabilities.max(axis=1)

confidence_results = cleaned_results.copy()
confidence_results["confidence"] = max_confidence

print("Average confidence for correct predictions:")
print(
    confidence_results.loc[
        confidence_results["correct"], "confidence"
    ].mean().round(4)
)

print("\nAverage confidence for incorrect predictions:")
print(
    confidence_results.loc[
        ~confidence_results["correct"], "confidence"
    ].mean().round(4)
)

Average confidence for correct predictions:
0.8507

Average confidence for incorrect predictions:
0.5425


In [39]:
import shap
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# 1. Extract TF-IDF and Classifier from your cleaned_model
tfidf = cleaned_model.named_steps["tfidf"]
classifier = cleaned_model.named_steps["classifier"]

X_train_tfidf = tfidf.transform(train_df["text"])
X_test_tfidf = tfidf.transform(test_df["text"])
feature_names = tfidf.get_feature_names_out()
categories = list(classifier.classes_)

# 2. Initialize SHAP LinearExplainer
explainer = shap.LinearExplainer(classifier, X_train_tfidf)
shap_values = explainer.shap_values(X_test_tfidf)

# Ensure shape is (n_classes, n_samples, n_features)
shap_arr = np.array(shap_values)
if shap_arr.shape[0] != len(categories):
    shap_arr = np.transpose(shap_arr, (2, 0, 1))

# 3. FIXED: Use explainer.expected_value (SHAP's exact internal baseline)
base_values = np.array(explainer.expected_value)
shap_sum_scores = base_values[:, None] + shap_arr.sum(axis=2)
actual_model_scores = classifier.decision_function(X_test_tfidf).T

shap_error = np.abs(shap_sum_scores - actual_model_scores).max()
shap_faithfulness_accuracy = max(0.0, (1.0 - shap_error) * 100.0)

print("==========================================")
print("FINAL MODEL & EXPLAINABILITY SUMMARY")
print("==========================================")
print(f"Model Test Accuracy       : {accuracy_score(y_test, cleaned_pred) * 100:.2f}%")
print(f"Model Macro F1-Score      : {f1_score(y_test, cleaned_pred, average='macro'):.4f}")
print(f"SHAP Explainability Acc.  : {shap_faithfulness_accuracy:.2f}% (Error: {shap_error:.2e})")
print("==========================================\n")

# 4. Explain a Single Test Ticket using SHAP
sample_index = 4
sample_ticket = test_df.iloc[sample_index]["text"]
pred_category = cleaned_model.predict([sample_ticket])[0]
pred_class_idx = categories.index(pred_category)

print("Sample Ticket:", sample_ticket)
print("Actual Category:", test_df.iloc[sample_index]["category_truth"])
print("Predicted Category:", pred_category)
print("\nTop SHAP Word Contributions for this ticket:")

present_words_mask = X_test_tfidf[sample_index].toarray()[0] > 0
present_indices = np.where(present_words_mask)[0]
ticket_shap = shap_arr[pred_class_idx, sample_index, present_indices]

top_order = np.argsort(ticket_shap)[::-1][:8]
for i in top_order:
    word = feature_names[present_indices[i]]
    score = ticket_shap[i]
    print(f"  {word:20s} : {score:+.4f}")

C:\Users\MIDHUN NAIR R I\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FINAL MODEL & EXPLAINABILITY SUMMARY
Model Test Accuracy       : 80.52%
Model Macro F1-Score      : 0.7630
SHAP Explainability Acc.  : 100.00% (Error: 2.00e-14)

Sample Ticket: File Share Access - [TICKET ID] - [NAME] ([COMPANY]. [LOCATION])Service Request: File Share Access     Grant access to file share for the following user:  [NAME] ([COMPANY]. [LOCATION]) (Username: [USERNAME])  Access Type: (RW) Read / Write  File Share:  J:/CDC-C     Requestor: [NAME] ([COMPANY]. [LOCATION])  Comment:  Frau [USERNAME] wird ab 7. Januar im Raum 2.36 ([ID] Flügel 4) als Azubi tätig werden. Entstehende Kosten zu Lasten 47006.
Actual Category: Fileservice
Predicted Category: Fileservice

Top SHAP Word Contributions for this ticket:
  file                 : +0.4374
  share                : +0.3663
  file share           : +0.3663
  access               : +0.1460
  share access         : +0.1423
  to file              : +0.1382
  following user       : +0.1104
  comment              : +0.1104


In [40]:
import joblib
joblib.dump(cleaned_model, "incident_triage_model.pkl")
print("Saved incident_triage_model.pkl successfully!")

Saved incident_triage_model.pkl successfully!
